0100000001c997a5e56e104102fa209c6a852dd90660a20b2d9c352423edce25857fcd3704000000004847304402204e45e16932b8af514961a1d3a1a25fdf3f4f7732e9d624c6c61548ab5fb8cd410220181522ec8eca07de4860a4acdd12909d831cc56cbbac4622082221a8768d1d0901ffffffff0200ca9a3b00000000434104ae1a62fe09c5f51b13905f07f06b99a2f7159b2225f374cd378d71302fa28414e7aab37397f554a7df5f142c21c1b7303b8a0626f1baded5c72a704f7e6cd84cac00286bee0000000043410411db93e1dcdb8a016b49840f8c53bc1eb68a382e97b1482ecad7b148a6909a5cb2e0eaddfb84ccf9744464f82e160bfa9b8b64f9d4c03f999b8643f656b412a3ac00000000



helpers.py

In [27]:
def varint2int(reader):
    first = int.from_bytes(reader.read(1))
    if first <= 252:
        return first
    if first == 253:
        return int.from_bytes(reader.read(2)[::-1])
    if first == 254:
        return int.from_bytes(reader.read(4)[::-1])
    if first == 255:
        return int.from_bytes(reader.read(8)[::-1])

if __name__ == "__main__":
    from io import BytesIO
    
    h = 'fd0302'
    
    reader = BytesIO(bytes.fromhex(h))
    print(varint2int(reader))

515


In [28]:
def int2varint(n):
    if n < 0 or n >= 2**64:
        return None
    if n <= 252:
        return n.to_bytes(1, 'little')
    if n <= 2**16:
        return int(253).to_bytes(1,'little') + n.to_bytes(2,'little')
    if n <= 2**32:
        return int(254).to_bytes(1,'little') + n.to_bytes(4,'little')
    if n <= 2**64:
        return int(255).to_bytes(1,'little') + n.to_bytes(8,'little')
    
print (int2varint(515).hex())

fd0302


Transaction.py

In [29]:

class TxOut:
    def __init__(self, value, script_pk):
        self.value = value
        self.script_pk = script_pk

@classmethod
def parse(cls, reader):
    value = reader.read(8)
    script_len = helpers.varint2int(reader)
    script_pk = reader.read(script_len)
    return cls(value, script_pk)

tests.py

In [ ]:
import secrets
from io import BytesIO
#testing txout

value = secrets.token_bytes(4) + 4*b'\x00'
n = 515
script_len = helpers.int2varint(n)
script_pk = secrets.token_bytes(n)

bs = value + script_len + script_pk
tx_out = transaction.TxOut.parse(reader)



class TxIn:
    def __init__(self, prev_tx, prev_index, script_sig, sequence):
        self.prev_tx = prev_tx
        self.prev_index = prev_index
        self.script_sig = script_sig
        self.sequence = sequence

    @classmethod
    def parse(cls, reader):
        prev_tx = reader.read(32)
        prev_index = int.from_bytes(reader.read(4), 'little')
        script_len = helpers.varint2int(reader)
        script_sig = reader.read(script_len)
        sequence = reader.read(4)
        return cls(prev_tx, prev_index, script_sig, sequence)

AttributeError: type object 'TxOut' has no attribute 'parse'